# Avengers Roster Analytics - Python Preparation Notebook
**Goal:** read the cleaned SQL table (`dbo.avengers_clean`), run final checks, add the columns we need for visualization (segments, decades, labels), and save the final dataset.

| Part | What happens |
|---|---|
| 1 | Setup and settings |
| 2 | Connect to SQL Server and load the data |
| 3 | Inspect and validate the data |
| 4 | Fix data types |
| 5 | Create new columns (feature engineering) |
| 6 | Assign the 6 member segments from the BRD |
| 7 | Final quality check |
| 8 | Summary tables (preview of dashboard numbers) |
| 9 | Save the final dataset (CSV and SQL Server) |

## 1. Setup and settings

In [ ]:
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from sqlalchemy import create_engine

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---------- SETTINGS: change these to match your computer ----------
USE_SQL       = True                       # True = read from SQL Server, False = read the CSV file
SERVER        = r"localhost\SQLEXPRESS"    # copy the exact "Server name" from the SSMS login window
DATABASE      = "Avengers_Analytics"
DRIVER        = "ODBC Driver 17 for SQL Server"   # or "ODBC Driver 18 for SQL Server"
CSV_FALLBACK  = "avengers_clean.csv"       # only used when USE_SQL = False
SNAPSHOT_YEAR = 2015                       # same snapshot year as the BRD
# --------------------------------------------------------------------

## 2. Connect to SQL Server and load the data
**One-time install** (run in a terminal or Anaconda Prompt): `pip install pandas sqlalchemy pyodbc`
You also need the *ODBC Driver for SQL Server* installed on Windows. To see which one you have, run `import pyodbc; print(pyodbc.drivers())`.

In [ ]:
def get_engine():
    """Create a connection to SQL Server using your Windows login."""
    conn_str = (
        f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};"
        "Trusted_Connection=yes;TrustServerCertificate=yes;"
    )
    return create_engine("mssql+pyodbc:///?odbc_connect=" + quote_plus(conn_str))


if USE_SQL:
    engine = get_engine()
    df_raw = pd.read_sql("SELECT * FROM dbo.avengers_clean", engine)
    print("Loaded from SQL Server")
else:
    df_raw = pd.read_csv(CSV_FALLBACK)
    print("Loaded from CSV")

df = df_raw.copy()                       # work on a copy, keep df_raw untouched
df.columns = df.columns.str.lower()      # simple lowercase column names
print(df.shape)
df.head()

## 3. Inspect and validate the data

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# Each check must pass. If one fails, Python stops and shows which one.
assert len(df) == 173,                                   "Row count changed"
assert df["member_id"].is_unique,                        "Duplicate member_id"
assert df["wiki_url"].is_unique,                         "Duplicate wiki_url"
assert df["gender"].isin(["MALE", "FEMALE"]).all(),      "Unexpected gender value"
assert df["membership_type"].isin(["Full", "Honorary", "Academy", "Probationary"]).all(), "Unexpected membership type"
assert df["is_current"].isin([0, 1]).all(),              "is_current must be 0 or 1"
assert (df["appearances"] > 0).all(),                    "Appearances must be positive"
assert (df["return_count"] <= df["death_count"]).all(),  "Returns cannot be more than deaths"
print("All validation checks passed")

## 4. Fix data types

In [ ]:
df["is_current"] = df["is_current"].astype(int)
df["join_year"] = df["join_year"].astype("Int64")        # whole number that allows blanks
df["tenure_years"] = df["tenure_years"].astype("Int64")
for col in ["gender", "membership_type"]:
    df[col] = df[col].str.title()                         # MALE -> Male
df.dtypes

## 5. Create new columns (feature engineering)

In [ ]:
# 5a. Simple labels that are easier to read in charts
df["status"] = np.where(df["is_current"] == 1, "Active", "Former")

# 5b. Decade of joining (1960s, 1970s ...) and tenure group
df["join_decade"] = df["join_year"].map(lambda y: "Unknown" if pd.isna(y) else f"{int(y) // 10 * 10}s")

tenure_bins   = [-1, 5, 10, 20, 40, 100]
tenure_labels = ["0-5 yrs", "6-10 yrs", "11-20 yrs", "21-40 yrs", "40+ yrs"]
df["tenure_group"] = (pd.cut(df["tenure_years"].astype(float), bins=tenure_bins, labels=tenure_labels)
                        .astype(object).fillna("Unknown"))

# 5c. Appearance ranking and share of the total
df["appearance_rank"] = df["appearances"].rank(ascending=False, method="min").astype(int)
df["appearance_share_pct"] = (df["appearances"] / df["appearances"].sum() * 100).round(2)
top_cut = df["appearances"].quantile(0.75)               # top quartile starts here
df["is_top_quartile"] = (df["appearances"] >= top_cut).astype(int)
print("Top quartile starts at", top_cut, "appearances")

# 5d. Life-and-death label
df["death_status"] = np.select(
    [
        df["ever_died"] == 0,
        (df["ever_died"] == 1) & (df["return_count"] == 0),
        (df["return_count"] > 0) & (df["unreturned_deaths"] == 0),
        (df["return_count"] > 0) & (df["unreturned_deaths"] > 0),
    ],
    ["Never Died", "Died - No Return", "Died & Returned", "Returned, Then Died Again"],
    default="Other",
)
df["death_status"].value_counts()

## 6. Assign the 6 member segments (BRD section 7)
Each character gets **exactly one** segment. The rules are checked **in this order** (first match wins):

1. **Core Veterans** - Full member, active, top-quartile appearances, tenure over 10 years
2. **Resurrected** - died at least once and returned
3. **Reserve & Academy** - Honorary, Academy or Probationary member
4. **Former Members** - not currently active
5. **New Recruits** - active, joined within the last 10 years
6. **Active Regulars** - all other active members

In [ ]:
tenure = df["tenure_years"].fillna(-1).astype(int)       # unknown tenure = -1, so it never counts as "long" or "new"

conditions = [
    (df["membership_type"] == "Full") & (df["is_current"] == 1) & (df["is_top_quartile"] == 1) & (tenure > 10),
    (df["is_resurrected"] == 1),
    (df["membership_type"] != "Full"),
    (df["is_current"] == 0),
    (df["is_current"] == 1) & (tenure >= 0) & (tenure <= 10),
]
names = ["Core Veterans", "Resurrected", "Reserve & Academy", "Former Members", "New Recruits"]

df["segment"] = np.select(conditions, names, default="Active Regulars")
df["segment"].value_counts()

## 7. Final quality check

In [ ]:
assert df["segment"].notna().all(),                  "Some members have no segment"
assert df["segment"].nunique() == 6,                 "Expected 6 segments"
assert df["segment"].value_counts().sum() == len(df),"Each member must be in exactly one segment"
assert not df["member_id"].duplicated().any(),       "Duplicate member"
print("Final checks passed. Rows:", len(df), "| Columns:", df.shape[1])

## 8. Summary tables (preview of the dashboard numbers)

In [ ]:
# 8a. Segment summary
seg = (df.groupby("segment")
         .agg(members=("member_id", "count"),
              active=("is_current", "sum"),
              total_appearances=("appearances", "sum"),
              avg_appearances=("appearances", "mean"))
         .round(1))
seg["members_pct"] = (seg["members"] / seg["members"].sum() * 100).round(1)
seg["appearance_share_pct"] = (seg["total_appearances"] / seg["total_appearances"].sum() * 100).round(1)
seg.sort_values("members", ascending=False)

In [ ]:
# 8b. Mortality and resurrection by membership type
mort = (df.groupby("membership_type")
          .agg(members=("member_id", "count"),
               died=("ever_died", "sum"),
               resurrected=("is_resurrected", "sum")))
mort["death_rate_pct"] = (mort["died"] / mort["members"] * 100).round(1)
mort["return_rate_pct"] = (mort["resurrected"] / mort["died"] * 100).round(1)
mort

In [ ]:
# 8c. Recruitment by decade and gender split
print(df.groupby("join_decade").size().rename("members_joined"))
print()
print(df.groupby("gender").agg(members=("member_id", "count"),
                               avg_appearances=("appearances", "mean"),
                               death_rate=("ever_died", "mean")).round(2))

In [ ]:
# 8d. Top 10 most prominent members
df.nsmallest(10, "appearance_rank")[["appearance_rank", "name_alias", "appearances",
                                      "appearance_share_pct", "segment", "status"]]

## 9. Save the final dataset

In [ ]:
final_cols = ["member_id", "name_alias", "gender", "membership_type", "status", "is_current",
              "appearances", "appearance_rank", "appearance_share_pct", "is_top_quartile",
              "join_year", "join_decade", "tenure_years", "tenure_group", "join_year_missing",
              "is_probationary", "death_count", "return_count", "unreturned_deaths",
              "ever_died", "is_multi_death", "is_resurrected", "death_status",
              "segment", "name_imputed", "wiki_url", "notes"]
df_final = df[final_cols].copy()

# 9a. Save as CSV (opens in Excel and Power BI)
df_final.to_csv("avengers_final.csv", index=False)

# 9b. Save back into SQL Server as a new table: dbo.avengers_final
if USE_SQL:
    df_final.to_sql("avengers_final", engine, schema="dbo", if_exists="replace", index=False)
    print("Saved to SQL Server table dbo.avengers_final")

print("Saved avengers_final.csv with", df_final.shape[0], "rows and", df_final.shape[1], "columns")
df_final.head()

### Next step: visualization
In **Power BI Desktop**: *Home -> Get data -> SQL Server* -> enter your server name and database `Avengers_Analytics` -> choose `dbo.avengers_final`. (Or *Get data -> Text/CSV* -> `avengers_final.csv`.)
Suggested pages: **Home**, **Roster Overview** (segments, appearances, recruitment by decade), **Mortality & Resurrection** (death status, return rate).